# Лабораторная работа 4. Вариант 1
**Тема.** Задачи на минимизацию с параллельными станками

## **Задача 1**
$$
P_3 | \space | C_{max}
$$ 
Класс задач с параллельными станками ($P_m$) отличается от задач одним станком ($1$) наличием возможности параллельно выполнять работы. В остальном здесь присутствуют все те же задачи, например на минимизацию времени выполнения. 

### Алгоритм
В данном случае для решения таких задач применяется эвристика **LPT** [1, 3.1.1]. Данную эвристику можно сформулировать так:
> Сначала выполняй самые длинные работы

То есть речь идет опять о сортировке по времени выполнения. 

In [1]:
jobs = {
    'J1': 7, 
    'J2': 8, 
    'J3': 4, 
    'J4': 9, 
    'J5': 12, 
    'J6': 5, 
    'J7': 3, 
    'J8': 9, 
    'J9': 5, 
    'J10': 12, 
    'J11': 7, 
    'J12': 5, 
    'J13': 8
}

Проведем сортировку входящего набора работ

In [2]:
sorted_jobs = sorted(jobs.items(), key=lambda x: x[1], reverse=True)

machines = [0, 0, 0]
schedule = {0: [], 1: [], 2: []}

Смоделируем процесс выполнения работ

In [3]:
for j_id, p in sorted_jobs:
    m_idx = machines.index(min(machines))
    
    start = machines[m_idx]
    end = start + p
    machines[m_idx] = end
    schedule[m_idx].append((j_id, start, end))

Выведем конечное расписание

In [4]:
res_sched, res_cmax = schedule, max(machines)

for m, ops in res_sched.items():
    print(f"Станок {m+1}: {ops}")
print(f"C_max: {res_cmax}")

Станок 1: [('J5', 0, 12), ('J2', 12, 20), ('J11', 20, 27), ('J3', 27, 31)]
Станок 2: [('J10', 0, 12), ('J13', 12, 20), ('J6', 20, 25), ('J9', 25, 30), ('J7', 30, 33)]
Станок 3: [('J4', 0, 9), ('J8', 9, 18), ('J1', 18, 25), ('J12', 25, 30)]
C_max: 33


## **Задача 2**
$$
P_3 | prmp | C_{max}
$$ 

### Алгоритм
Для решения задач на параллельных станках с прерыванием обычно применяют direct graph technique (правило Макнотона) [1, 3.3]. Первым этапом реализации правила Макнотона является высянение теоретически доступного минимума $C_{max}$. Выполняется это следующим образом:
$$
C^{*}_{max} = max(max_{j}p_{j}, \frac{1}{m}\cdot \sum_{j=1}^{n}p_j)
$$

In [5]:
total_p = sum(jobs.values())
max_p = max(jobs.values())
c_max = max(max_p, total_p / m)

То есть мы всегда упремся в самую длинную работу $T = C^{*}_{max}$.
</br>
Далее выполняются следующие шаги:
- шаг 1: начинаем с первого станка и укладываем на него некоторую работу

In [6]:
schedule = {i: [] for i in range(1, m + 1)}
current_m = 1
current_t = 0

- шаг 2: если работа поместилась до $T$, то кладем ее следующей и повторяем шаг, иначе **шаг 3**;
- шаг 3: отрезаем ту часть, которая влезла до $T$, и оставляем её на текущем станке;
- шаг 4: переходим на следующий станок в момент времени $t=0$.

Выполняем для каждого станка. Смоделируем процесс.

In [7]:
for j_id, p in jobs.items():
    remaining_p = p
    while remaining_p > 0:
        capacity = c_max - current_t
        if remaining_p <= capacity:
            schedule[current_m].append((j_id, current_t, current_t + remaining_p))
            current_t += remaining_p
            remaining_p = 0
        else:
            schedule[current_m].append((j_id, current_t, c_max))
            remaining_p -= capacity
            current_m += 1
            current_t = 0

Выведем результат

In [8]:
c_max_val, final_sched =  c_max, schedule

print(f"Оптимальный C_max: {c_max_val:.2f}")
for m_id, pieces in final_sched.items():
    print(f"Станок {m_id}: {pieces}")

Оптимальный C_max: 47.00
Станок 1: [('J1', 0, 7), ('J2', 7, 15), ('J3', 15, 19), ('J4', 19, 28), ('J5', 28, 40), ('J6', 40, 45), ('J7', 45, 47.0)]
Станок 2: [('J7', 0, 1.0), ('J8', 1.0, 10.0), ('J9', 10.0, 15.0), ('J10', 15.0, 27.0), ('J11', 27.0, 34.0), ('J12', 34.0, 39.0), ('J13', 39.0, 47.0)]


## **Задача 3**
$$
Q_4 | prmp | C_{max}
$$ 
Задачи класса ($Q$) отличаются тем, что вводят скорость станков как переменную. То есть теперь станки не одинаковы, у них есть выраженная производительность.
### Алгоритм
Для решения задачи с неоднородными станками простое правило Макнотона не подойдет, тк оно предполагает, что работа может быть одинаково эффективно переложена на любой станок. У $Q$ это не так. Поэтому здесь применяется **LRPT-FM** [1, 3.7]
</br>
LRPT-FM базируется на двух принципах:
> - В любой момент времени мы смотрим на остаточное время выполнения всех работ. Чем оно больше, тем выше приоритет.
>
> - Самая длинная работа (по остатку) назначается на самый быстрый станок. Вторая по длине — на второй по скорости, и так далее.

При этом правило LRPT-FM является динамическим, то есть:
- Когда работа на быстром станке полностью готова. Тогда работа со второго станка переходит на освободившийся быстрый, с третьего — на второй и т.д.
- Если в процессе работы остаток времени выполнения одной задачи сравнялся с остатком другой, они начинают делить быстрый ресурс между собой (через бесконечно малые прерывания), чтобы уменьшаться синхронно.

Введем вектор скоростей станков.

In [9]:
speeds = [3, 4, 3, 2]

Подготовим внешние условия

In [10]:
p_rem = {j: float(p) for j, p in jobs.items()}
speeds = sorted(speeds, reverse=True)
n_m = len(speeds)

t = 0
dt = 0.01
history = []

Проведем моделирование процесса.

In [11]:
while sum(p_rem.values()) > 0.001:
    active_jobs = sorted([j for j in p_rem if p_rem[j] > 0], 
                        key=lambda x: p_rem[x], reverse=True)

    for i in range(min(len(active_jobs), n_m)):
        job_id = active_jobs[i]
        s = speeds[i]
        
        consumption = s * dt
        if p_rem[job_id] < consumption:
            consumption = p_rem[job_id]
        
        p_rem[job_id] -= consumption
        history.append((job_id, i+1, t, t + dt))
        
    t += dt
    if t > 1000: break

Выведем результат

In [12]:
final_cmax, _ = t, history
print(f"C_max для Q4 по правилу LRPT-FM: {final_cmax:.2f}")

C_max для Q4 по правилу LRPT-FM: 7.85


## **Задача 4**
Решить
$$
P_3 | | C_{max}
$$ 
 с помощью полиномиальной приближенной схемы с точностью $\epsilon = \frac{1}{5} $
### Алгоритм
Алгоритмы типа PTAS (полиномиальная приближенная схема) обвчно используют для приблеженного решения NP-полных задач. В данном случае допускается погрешность $\epsilon = \frac{1}{5} = 20%$

In [13]:
m = 3
epsilon = 1/5

Сначала необходимо определить критерий для больших работ:
$$
p_j > \epsilon \cdot LB
$$
где $LB$ -- нижняя граница $C_max$

In [14]:
total_load = sum(jobs.values())
max_job = max(jobs.values())
lb = max(total_load / m, max_job)
threshold = lb * epsilon
threshold

6.266666666666667

Поделим работы по формальному критерию

In [15]:
large_jobs = {k: v for k, v in jobs.items() if v > threshold}
small_jobs = {k: v for k, v in jobs.items() if v <= threshold}

Далее, большие работы вполне точно. Например, через фиксацию предполагаемого времени завершения.

In [16]:
machine_loads = [0] * m
schedule = [[] for _ in range(m)]

sorted_large = sorted(large_jobs.items(), key=lambda x: x[1], reverse=True)
for j_id, p in sorted_large:
    m_idx = machine_loads.index(min(machine_loads))
    machine_loads[m_idx] += p
    schedule[m_idx].append((j_id, p))

Малые работы распределяются по мере освобождения станков. Их отдельный вес не слишком влияет на конечный результат.

In [17]:
for j_id, p in small_jobs.items():
    m_idx = machine_loads.index(min(machine_loads))
    machine_loads[m_idx] += p
    schedule[m_idx].append((j_id, p))

Выведем результат

In [18]:
res_schedule, loads, lower_bound = schedule, machine_loads, lb

print(f"Нижняя граница (LB): {lower_bound:.2f}")
print(f"Целевой максимум (1+eps)*LB: {lower_bound * (1 + epsilon):.2f}")
print("-" * 30)

for i, load in enumerate(loads):
    job_list = ", ".join([f"{j[0]}({j[1]})" for j in res_schedule[i]])
    print(f"M{i+1} [Итого {load}]: {job_list}")

print("-" * 30)
print(f"Полученный C_max: {max(loads)}")

Нижняя граница (LB): 31.33
Целевой максимум (1+eps)*LB: 37.60
------------------------------
M1 [Итого 32]: J5(12), J2(8), J11(7), J9(5)
M2 [Итого 29]: J10(12), J13(8), J3(4), J6(5)
M3 [Итого 33]: J4(9), J8(9), J1(7), J7(3), J12(5)
------------------------------
Полученный C_max: 33


## References
1. ALGORITHMS FOR SEQUENCING AND SCHEDULING, Ibrahim M. Alharkan